# Bird Acoustics Classifier — Data Pipeline

End-to-end pipeline from raw audio to training-ready spectrogram images.

| Step | Description |
|------|-------------|
| 1 | **Setup** — environment detection (local / Colab) |
| 2 | **Configuration** — load parameters from `config/default.yaml` |
| 3 | **Download** — fetch `.mp3` recordings from Xeno-canto API |
| 4 | **Preprocessing** — convert audio to mel-spectrogram PNG tiles |
| 5 | **Summary** — dataset statistics and visual inspection |

> **Run All** (`Runtime → Run all` on Colab, `Kernel → Restart & Run All` locally) executes the full pipeline in one shot.

---
## 1. Setup

In [ ]:
import sys, os

BRANCH = "claude/setup-project-structure-lVRTH"

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

    repo = '/content/bird-acoustics-classifier'
    if not os.path.exists(repo):
        !git clone -q -b {BRANCH} https://github.com/danort92/bird-acoustics-classifier.git {repo}
    else:
        !git -C {repo} fetch -q origin {BRANCH} && git -C {repo} checkout -q {BRANCH} && git -C {repo} pull -q
    %cd /content/bird-acoustics-classifier
    !pip install -q -r requirements.txt

    # Symlink data dirs to Drive so files survive session restarts
    for subdir in ['data/raw', 'data/processed']:
        drive_path = f'/content/drive/MyDrive/bird-acoustics-classifier/{subdir}'
        os.makedirs(drive_path, exist_ok=True)
        if os.path.islink(subdir) or os.path.isdir(subdir):
            !rm -rf {subdir}
        !ln -s {drive_path} {subdir}

    sys.path.insert(0, repo)
    print(f'Colab ready (branch: {BRANCH}). Data will be saved to Google Drive.')
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    sys.path.insert(0, os.getcwd())
    print('Local environment. Working directory:', os.getcwd())

---
## 2. Configuration

Parameters are loaded from `config/default.yaml`. Override any value in the cell below.

### Available species — Alpine zone (Italy / Austria / Switzerland)

| # | Scientific name | Common name | Habitat |
|---|----------------|-------------|---------|
| 1 | *Turdus torquatus* | Ring ouzel | Rocky slopes, high-altitude forests |
| 2 | *Phoenicurus ochruros* | Black redstart | Rocky terrain, mountain villages |
| 3 | *Prunella collaris* | Alpine accentor | High rocky areas above treeline |
| 4 | *Pyrrhocorax graculus* | Yellow-billed chough | Alpine cliffs and glaciers |
| 5 | *Pyrrhocorax pyrrhocorax* | Red-billed chough | Alpine meadows and cliffs |
| 6 | *Tichodroma muraria* | Wallcreeper | Vertical rock faces |
| 7 | *Anthus spinoletta* | Water pipit | Alpine meadows and streams |
| 8 | *Montifringilla nivalis* | White-winged snowfinch | Above treeline, snowfields |
| 9 | *Lagopus muta* | Rock ptarmigan | High alpine tundra |
| 10 | *Dryocopus martius* | Black woodpecker | Subalpine conifer forests |
| 11 | *Tetrao urogallus* | Western capercaillie | Old-growth conifer forests |
| 12 | *Picoides tridactylus* | Three-toed woodpecker | Spruce forests |
| 13 | *Loxia curvirostra* | Common crossbill | Conifer forests |
| 14 | *Nucifraga caryocatactes* | Spotted nutcracker | Mountain conifer forests |
| 15 | *Regulus ignicapilla* | Firecrest | Mixed mountain forests |
| 16 | *Cinclus cinclus* | White-throated dipper | Alpine streams and torrents |
| 17 | *Ficedula albicollis* | Collared flycatcher | Deciduous mountain forests |
| 18 | *Saxicola rubetra* | Whinchat | Subalpine meadows |
| 19 | *Emberiza cia* | Rock bunting | Rocky slopes with sparse vegetation |
| 20 | *Gypaetus barbatus* | Bearded vulture | High alpine cliffs (reintroduced) |

To use a subset, uncomment `SPECIES` in the cell below and remove the lines you don't want.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import yaml

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

# ── Load defaults from config/default.yaml ────────────────────────────────────
with open('config/default.yaml') as f:
    cfg = yaml.safe_load(f)

# ── Override here — None = use value from default.yaml ────────────────────────

# Species to download: None = all 20 Alpine species from config/default.yaml
# To use a subset, uncomment SPECIES and remove the lines you don't want
SPECIES = None
# SPECIES = [
#     "Turdus torquatus",         # Ring ouzel              — rocky slopes, high-altitude forests
#     "Phoenicurus ochruros",     # Black redstart          — rocky terrain, mountain villages
#     "Prunella collaris",        # Alpine accentor         — high rocky areas above treeline
#     "Pyrrhocorax graculus",     # Yellow-billed chough    — alpine cliffs and glaciers
#     "Pyrrhocorax pyrrhocorax",  # Red-billed chough       — alpine meadows and cliffs
#     "Tichodroma muraria",       # Wallcreeper             — vertical rock faces
#     "Anthus spinoletta",        # Water pipit             — alpine meadows and streams
#     "Montifringilla nivalis",   # White-winged snowfinch  — above treeline, snowfields
#     "Lagopus muta",             # Rock ptarmigan          — high alpine tundra
#     "Dryocopus martius",        # Black woodpecker        — subalpine conifer forests
#     "Tetrao urogallus",         # Western capercaillie    — old-growth conifer forests
#     "Picoides tridactylus",     # Three-toed woodpecker   — spruce forests
#     "Loxia curvirostra",        # Common crossbill        — conifer forests
#     "Nucifraga caryocatactes",  # Spotted nutcracker      — mountain conifer forests
#     "Regulus ignicapilla",      # Firecrest               — mixed mountain forests
#     "Cinclus cinclus",          # White-throated dipper   — alpine streams and torrents
#     "Ficedula albicollis",      # Collared flycatcher     — deciduous mountain forests
#     "Saxicola rubetra",         # Whinchat                — subalpine meadows
#     "Emberiza cia",             # Rock bunting            — rocky slopes with sparse vegetation
#     "Gypaetus barbatus",        # Bearded vulture         — high alpine cliffs (reintroduced)
# ]

# Max recordings per species: None = 100 (from config)
MAX_PER_SPECIES = None
# MAX_PER_SPECIES = 50

# Quality filter: None = "all" grades (from config); set "A" for best quality only (fewer files)
QUALITY = None
# QUALITY = "A"

# Country filter: None = worldwide
COUNTRIES = None
# COUNTRIES = ["Italy", "Austria", "Switzerland"]

# Audio parameters: None = use value from config/default.yaml
SAMPLE_RATE    = None  # default: 22050 Hz
CLIP_DURATION  = None  # default: 5.0 s
CLIP_OVERLAP   = None  # default: 0.0 (no overlap)
N_MELS         = None  # default: 128
N_FFT          = None  # default: 2048
HOP_LENGTH     = None  # default: 512
F_MIN          = None  # default: 500.0 Hz
F_MAX          = None  # default: 15000.0 Hz
TOP_DB         = None  # default: 80.0 dB
IMG_SIZE       = None  # default: (224, 224) px

# ── Resolve: override wins over YAML default ──────────────────────────────────
d, a = cfg['download'], cfg['audio']

SPECIES         = SPECIES         or cfg['species']
MAX_PER_SPECIES = MAX_PER_SPECIES or d['max_per_species']
QUALITY         = QUALITY         or d['quality']
COUNTRIES       = COUNTRIES       or d.get('countries') or None
RAW_DIR         = cfg['data']['raw_dir']
PROCESSED_DIR   = cfg['data']['processed_dir']

from src.preprocessing import AudioConfig
AUDIO_CONFIG = AudioConfig(
    sample_rate   = SAMPLE_RATE   or a['sample_rate'],
    clip_duration = CLIP_DURATION or a['clip_duration'],
    clip_overlap  = CLIP_OVERLAP  if CLIP_OVERLAP is not None else a['clip_overlap'],
    n_mels        = N_MELS        or a['n_mels'],
    n_fft         = N_FFT         or a['n_fft'],
    hop_length    = HOP_LENGTH    or a['hop_length'],
    f_min         = F_MIN         or a['f_min'],
    f_max         = F_MAX         or a['f_max'],
    top_db        = TOP_DB        or a['top_db'],
    img_size      = IMG_SIZE      or tuple(a['img_size']),
)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"{'─'*50}")
print(f"  Species ({len(SPECIES)}):")
for s in SPECIES:
    print(f"    • {s}")
print(f"{'─'*50}")
print(f"  Max/species : {MAX_PER_SPECIES}  |  Quality: {QUALITY}  |  Countries: {COUNTRIES or 'worldwide'}")
print(f"  Sample rate : {AUDIO_CONFIG.sample_rate} Hz  |  Clip: {AUDIO_CONFIG.clip_duration}s  |  Overlap: {AUDIO_CONFIG.clip_overlap}")
print(f"  Mel bins    : {AUDIO_CONFIG.n_mels}  |  FFT: {AUDIO_CONFIG.n_fft}  |  Hop: {AUDIO_CONFIG.hop_length}")
print(f"  Freq range  : {AUDIO_CONFIG.f_min}–{AUDIO_CONFIG.f_max} Hz  |  Top dB: {AUDIO_CONFIG.top_db}")
print(f"  Image size  : {AUDIO_CONFIG.img_size[0]}×{AUDIO_CONFIG.img_size[1]} px")
print(f"{'─'*50}")

---
## 3. Download

Downloads `.mp3` recordings from the [Xeno-canto API v3](https://xeno-canto.org/explore/api).

> **API key required** since October 2025. Get a free key at [xeno-canto.org/article/854](https://xeno-canto.org/article/854) and set it as:
> ```bash
> export XENO_CANTO_API_KEY="your_key"
> ```
> If not set, you will be prompted interactively below.

Already-downloaded files are skipped automatically — safe to re-run.

In [ ]:
import pandas as pd
from src.download import XenoCantoDownloader

downloader = XenoCantoDownloader(output_dir=RAW_DIR)

results = downloader.download_species(
    species_list    = SPECIES,
    max_per_species = MAX_PER_SPECIES,
    quality         = QUALITY,
    countries       = COUNTRIES,
)

# Summary table
rows = [{'species': sp, 'downloaded': len(paths)} for sp, paths in results.items()]
df_dl = pd.DataFrame(rows)
print(f"\nTotal downloaded: {df_dl['downloaded'].sum()} files across {len(df_dl)} species")
df_dl

In [ ]:
# Verify files on disk — only the species selected in section 2
configured = {s.replace(' ', '_').lower() for s in SPECIES}
downloaded = downloader.list_downloaded()

summary_rows = []
for species_dir, files in downloaded.items():
    if species_dir.lower() not in configured:
        continue
    total_mb = sum(f.stat().st_size for f in files) / 1_048_576
    summary_rows.append({'species': species_dir, 'files': len(files), 'MB': round(total_mb, 2)})

df_disk = pd.DataFrame(summary_rows)
print(df_disk.to_string(index=False))
print(f"\nTotal: {df_disk['files'].sum()} files — {df_disk['MB'].sum():.1f} MB")

---
## 4. Preprocessing

Each `.mp3` recording is:
1. Loaded and resampled to `sample_rate` Hz
2. Sliced into `clip_duration`-second windows
3. Converted to a log-amplitude mel spectrogram
4. Saved as a `img_size` PNG — ready for CNN training

In [ ]:
from src.preprocessing import SpectrogramConverter
from IPython.display import Audio, display
import librosa

converter  = SpectrogramConverter(output_dir=PROCESSED_DIR, config=AUDIO_CONFIG)
raw_path   = Path(RAW_DIR)
configured = {s.replace(' ', '_').lower() for s in SPECIES}

if not raw_path.exists():
    print(f'RAW_DIR not found: {raw_path.resolve()}')
    print('→ Run step 3 (Download) first.')
else:
    representatives = []
    for sp_dir in sorted(raw_path.iterdir()):
        if sp_dir.is_dir() and sp_dir.name.lower() in configured:
            mp3s = sorted(sp_dir.glob('*.mp3'))
            if mp3s:
                representatives.append((sp_dir.name, mp3s[0]))

    missing = configured - {name.lower() for name, _ in representatives}
    if missing:
        print(f'Warning: no .mp3 files for: {", ".join(sorted(missing))}')
        print('→ Re-run step 3 to download missing species.\n')

    if not representatives:
        print(f'No .mp3 files found in {raw_path.resolve()}')
        print('→ Run step 3 (Download) first.')
    else:
        clip_samples = int(AUDIO_CONFIG.clip_duration * AUDIO_CONFIG.sample_rate)
        COLS = 4

        try:
            import ipywidgets as _widgets
            _has_widgets = True
        except ImportError:
            _has_widgets = False

        # Process species in rows of COLS
        for row_start in range(0, len(representatives), COLS):
            batch = representatives[row_start:row_start + COLS]
            n = len(batch)

            # ── Row of spectrograms ───────────────────────────────────────
            fig, axes = plt.subplots(1, n, figsize=(5 * n, 3))
            if n == 1:
                axes = [axes]
            for ax, (name, mp3) in zip(axes, batch):
                try:
                    converter.plot_spectrogram(mp3, clip_index=0, ax=ax,
                                               title=name.replace('_', ' '))
                except Exception as e:
                    ax.text(0.5, 0.5, str(e), ha='center', va='center',
                            transform=ax.transAxes, fontsize=8)
            fig.tight_layout()
            plt.show()

            # ── Row of audio players (aligned with spectrograms above) ────
            if _has_widgets:
                cell_w = f'{100 // n}%'
                outputs = []
                for name, mp3 in batch:
                    out = _widgets.Output(
                        layout=_widgets.Layout(width=cell_w, overflow='hidden'))
                    with out:
                        try:
                            y, sr = librosa.load(str(mp3),
                                                 sr=AUDIO_CONFIG.sample_rate, mono=True)
                            display(Audio(y[:clip_samples], rate=sr))
                        except Exception as e:
                            print(f'(unavailable: {e})')
                    outputs.append(out)
                display(_widgets.HBox(outputs))
            else:
                # Fallback when ipywidgets is not installed
                for name, mp3 in batch:
                    try:
                        y, sr = librosa.load(str(mp3),
                                             sr=AUDIO_CONFIG.sample_rate, mono=True)
                        print(f'  {name.replace("_", " ")}')
                        display(Audio(y[:clip_samples], rate=sr))
                    except Exception as e:
                        print(f'  {name}: (unavailable: {e})')

In [ ]:
# Run full preprocessing pipeline
# Set OVERWRITE = True to regenerate existing images
OVERWRITE = False

summary = converter.process_all(input_dir=RAW_DIR, overwrite=OVERWRITE, species=SPECIES)

print('\nClips per species:')
for species, n_clips in sorted(summary.items()):
    print(f'  {species:<40} {n_clips:>4} clips')
print(f'\nTotal clips: {sum(summary.values())}')

---
## 5. Dataset summary

In [ ]:
processed_path = Path(PROCESSED_DIR)
configured = {s.replace(' ', '_').lower() for s in SPECIES}

species_counts = {}
for d in sorted(processed_path.iterdir()):
    if d.is_dir() and d.name.lower() in configured:
        pngs = list(d.glob('*.png'))
        if pngs:
            species_counts[d.name] = len(pngs)
total = sum(species_counts.values())

clip_dur_str = f"{AUDIO_CONFIG.clip_duration:g}"
print(f'Species : {len(species_counts)}')
print(f'Clips   : {total}')
print(f'Img size: {AUDIO_CONFIG.img_size[0]}×{AUDIO_CONFIG.img_size[1]} px')
print(f'Duration: {clip_dur_str}s per clip @ {AUDIO_CONFIG.sample_rate} Hz')

if species_counts:
    fig, ax = plt.subplots(figsize=(max(8, len(species_counts) * 0.7), 5))
    names  = [n.replace('_', '\n') for n in species_counts]
    counts = list(species_counts.values())
    bars = ax.bar(range(len(names)), counts, color='steelblue', edgecolor='white')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, fontsize=7)
    ax.set_ylabel('Number of clips')
    ax.set_title(f'Clip distribution — {len(species_counts)} species, {total} total')
    ax.bar_label(bars, fontsize=7, padding=2)
    fig.tight_layout()
    plt.show()

In [ ]:
# Random sample of processed spectrogram images + audio clips
import random
import re
from PIL import Image
import librosa
from IPython.display import Audio, display, HTML

processed_path = Path(PROCESSED_DIR)
raw_path = Path(RAW_DIR)
configured = {s.replace(' ', '_').lower() for s in SPECIES}

species_counts = {}
png_files = []
for d in sorted(processed_path.iterdir()):
    if d.is_dir() and d.name.lower() in configured:
        pngs = list(d.glob('*.png'))
        if pngs:
            species_counts[d.name] = len(pngs)
            png_files.extend(pngs)

total = sum(species_counts.values())

if not png_files:
    print('No PNG files found — check that step 4 completed successfully.')
else:
    sample = random.sample(png_files, min(12, len(png_files)))
    cols = 4
    rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    for ax in np.array(axes).flatten():
        ax.set_visible(False)
    for ax, p in zip(np.array(axes).flatten(), sample):
        ax.set_visible(True)
        ax.imshow(Image.open(p))
        ax.set_title(p.parent.name.replace('_', ' '), fontsize=7)
        ax.axis('off')
    fig.suptitle('Random sample — processed spectrogram images', fontsize=11)
    fig.tight_layout()
    plt.show()

    # ── Corresponding audio clips ──────────────────────────────────────────
    clip_re      = re.compile(r'^(.+)_clip(\d+)$')
    clip_samples = int(AUDIO_CONFIG.clip_duration * AUDIO_CONFIG.sample_rate)
    step_samples = int(clip_samples * (1.0 - AUDIO_CONFIG.clip_overlap))

    display(HTML('<hr><b>Corresponding audio clips</b>'))
    for p in sample:
        m = clip_re.match(p.stem)
        if not m:
            continue
        stem, clip_idx = m.group(1), int(m.group(2))
        mp3_path = raw_path / p.parent.name / (stem + '.mp3')
        if not mp3_path.exists():
            display(HTML(f'<small>⚠ audio not found: {mp3_path.name}</small>'))
            continue
        try:
            y, sr = librosa.load(mp3_path, sr=AUDIO_CONFIG.sample_rate, mono=True)
            start = clip_idx * step_samples
            clip  = y[start : start + clip_samples]
            label = f'{p.parent.name.replace("_", " ")} — {stem} clip {clip_idx}'
            display(HTML(f'<small><b>{label}</b></small>'))
            display(Audio(clip, rate=sr))
        except Exception as exc:
            display(HTML(f'<small>⚠ {p.name}: {exc}</small>'))

    print(f'\nDataset ready: {total} images across {len(species_counts)} species in {PROCESSED_DIR}/')
    print('Next step: training notebook (coming).')

---
## 6. Training

Fine-tunes a pretrained **EfficientNet-B0** on the spectrograms produced in step 4.

| | |
|---|---|
| Model | EfficientNet-B0, ImageNet weights |
| Split | stratified 70 / 15 / 15 (train / val / test) |
| Optimiser | Adam + cosine LR annealing |
| Tracking | MLflow (`mlruns/`) |
| Checkpoint | `models/best_model.pt` (best val loss) |

> **Colab**: runtime must be set to **GPU** (Runtime → Change runtime type → T4 GPU).

In [ ]:
from src.model import TrainingConfig

train_cfg = TrainingConfig.from_yaml('config/default.yaml')

# The species list is inherited from section 2 (SPECIES variable)
# ── Optional overrides ────────────────────────────────────────────────────
# train_cfg.epochs        = 10
# train_cfg.batch_size    = 16
# train_cfg.learning_rate = 3e-4
# train_cfg.patience      = 5
# train_cfg.num_workers   = 2

# ── Section 6 = Baseline run (for comparison with section 7) ─────────────
# Uses the original augmentation strategy (RandomHorizontalFlip + ColorJitter)
# with no class balancing, no label smoothing, no progressive unfreezing.
# This mirrors the approach most tutorials use — and serves as the reference.
train_cfg.checkpoint_name      = 'best_model_baseline.pt'
train_cfg.augment_strategy     = 'basic'   # RandomHorizontalFlip + ColorJitter
train_cfg.label_smoothing      = 0.0
train_cfg.use_weighted_sampler = False
train_cfg.progressive_unfreeze = False

print(f"{'─'*52}")
print(f"  Model          : {train_cfg.model_name}")
print(f"  Epochs         : {train_cfg.epochs}  (patience={train_cfg.patience})")
print(f"  Batch size     : {train_cfg.batch_size}")
print(f"  Learning rate  : {train_cfg.learning_rate}")
print(f"  Val / Test     : {train_cfg.val_split} / {train_cfg.test_split}")
print(f"  Image size     : {train_cfg.img_size[0]}×{train_cfg.img_size[1]} px")
print(f"  Augmentation   : {train_cfg.augment_strategy}")
print(f"  Label smoothing: {train_cfg.label_smoothing}")
print(f"  Weighted sampl.: {train_cfg.use_weighted_sampler}")
print(f"  Prog. unfreeze : {train_cfg.progressive_unfreeze}")
print(f"  Output dir     : {train_cfg.output_dir}/{train_cfg.checkpoint_name}")
print(f"  MLflow URI     : {train_cfg.tracking_uri}")
print(f"{'─'*52}")

In [ ]:
import json, torch
from pathlib import Path
from src.model import BirdTrainer

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}'
      + (f' — {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else ''))

# ── Toggle ────────────────────────────────────────────────────────────────
FORCE_RETRAIN_BASELINE = False   # set True to retrain even if checkpoint exists
# ─────────────────────────────────────────────────────────────────────────

_ckpt = Path(train_cfg.output_dir) / train_cfg.checkpoint_name
_hist = _ckpt.with_suffix('.history.json')

if not FORCE_RETRAIN_BASELINE and _ckpt.exists():
    print(f'\nCheckpoint found — skipping baseline training.')
    print(f'  {_ckpt}')
    print(f'  Set FORCE_RETRAIN_BASELINE = True to retrain.\n')
    trainer          = BirdTrainer(train_cfg)
    _ckpt_data       = torch.load(_ckpt, map_location='cpu', weights_only=False)
    trainer.classes  = _ckpt_data['classes']
    baseline_history = json.loads(_hist.read_text()) if _hist.exists() else {
        'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': [],
        'test_loss': 0.0, 'test_acc': 0.0,
    }
    best_path = _ckpt
    print(f'Best model : {best_path}')
    print(f'Test acc   : {baseline_history.get("test_acc") or "n/a (run once to record)"}')
else:
    trainer                     = BirdTrainer(train_cfg)
    best_path, baseline_history = trainer.train(species=SPECIES)
    _hist.write_text(json.dumps(baseline_history))
    print(f'\nBest model : {best_path}')
    print(f'Test acc   : {baseline_history["test_acc"]:.3f}')

baseline_path    = best_path
baseline_trainer = trainer

In [ ]:
epochs_ran = list(range(1, len(history['train_loss']) + 1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_ran, history['train_loss'], label='train')
axes[0].plot(epochs_ran, history['val_loss'],   label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(epochs_ran, history['train_acc'], label='train')
axes[1].plot(epochs_ran, history['val_acc'],   label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1); axes[1].legend()

axes[2].plot(epochs_ran, history['lr'], color='green')
axes[2].set_title('Learning rate'); axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log')

fig.suptitle(
    f'Training curves — best val_loss={min(history["val_loss"]):.4f}  '
    f'test_acc={history["test_acc"]:.3f}',
    fontsize=12,
)
fig.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_true, y_pred = trainer.evaluate(best_path, species=SPECIES)

cm   = confusion_matrix(y_true, y_pred)
cm_n = cm.astype('float') / cm.sum(axis=1, keepdims=True)
n    = len(trainer.classes)
short = [c.replace('_', '\n') for c in trainer.classes]

fig, ax = plt.subplots(figsize=(max(10, n * 0.65), max(8, n * 0.55)))
im = ax.imshow(cm_n, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(short, fontsize=7, rotation=90)
ax.set_yticklabels(short, fontsize=7)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion matrix (normalised) — test acc={history["test_acc"]:.3f}')
for i in range(n):
    for j in range(n):
        v = cm_n[i, j]
        if v > 0.01:
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    fontsize=6, color='white' if v > 0.5 else 'black')
fig.tight_layout()
plt.show()

print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=trainer.classes, digits=3))

In [ ]:
import torch

ckpt = torch.load(best_path, map_location='cpu')
print(f"Checkpoint : {best_path}")
print(f"  Epoch    : {ckpt['epoch']}")
print(f"  Val loss : {ckpt['val_loss']:.4f}")
print(f"  Val acc  : {ckpt['val_acc']:.3f}")
print(f"  Classes  : {ckpt['num_classes']}")
print(f"  Model    : {ckpt['model_name']}")

try:
    from google.colab import files
    files.download(str(best_path))
    print('\nDownload started.')
except ImportError:
    print(f'\nTo run inference:')
    print(f'  from src.model import predict')
    print(f'  predict("path/to/spectrogram.png", "{best_path}")')

In [ ]:
import mlflow

mlflow.set_tracking_uri(train_cfg.tracking_uri)
client = mlflow.tracking.MlflowClient()
exp    = client.get_experiment_by_name(train_cfg.experiment_name)

if exp:
    runs = client.search_runs(
        exp.experiment_id,
        order_by=['metrics.val_loss ASC'],
        max_results=5,
    )
    print(f'Top 5 runs for "{train_cfg.experiment_name}":\n')
    for r in runs:
        m, p = r.data.metrics, r.data.params
        print(
            f"  {r.info.run_id[:8]}…  "
            f"val_loss={m.get('val_loss', float('nan')):.4f}  "
            f"val_acc={m.get('val_acc', float('nan')):.3f}  "
            f"test_acc={m.get('test_acc', float('nan')):.3f}  "
            f"lr={p.get('learning_rate')}"
        )
else:
    print('No runs found yet.')

print(f'\nMLflow UI: mlflow ui --backend-store-uri {train_cfg.tracking_uri}')

---
## 7. Augmentation experiment

Compares the **baseline** (section 6) against a fully improved training run
that adds four changes:

| | Baseline (§6) | Augmented (§7) |
|---|---|---|
| Augmentation | RandomHorizontalFlip + ColorJitter | **SpecAugment** (FreqMask + TimeMask + Noise + Erase) |
| Class balancing | uniform sampling | **WeightedRandomSampler** |
| Loss | CrossEntropy | CrossEntropy + **label smoothing 0.1** |
| Fine-tuning | all layers from epoch 1 | **progressive unfreeze** (head → backbone at epoch 5) |

> Results are compared in **§ 7c** with overlapping learning curves, a
> metric table, and side-by-side classification reports.

### 7a. Baseline recap

In [ ]:
# Baseline recap — results from section 6
print(f"{'─'*42}")
print(f"  Baseline training summary")
print(f"{'─'*42}")
print(f"  Augmentation   : basic (HFlip + ColorJitter)")
print(f"  Label smoothing: 0.0")
print(f"  Weighted sampl.: False")
print(f"  Prog. unfreeze : False")
print(f"{'─'*42}")
print(f"  Best val acc   : {max(baseline_history['val_acc']):.3f}  (epoch {baseline_history['val_acc'].index(max(baseline_history['val_acc']))+1})")
print(f"  Best val loss  : {min(baseline_history['val_loss']):.4f}")
print(f"  Test accuracy  : {baseline_history['test_acc']:.3f}")
print(f"  Epochs ran     : {len(baseline_history['train_loss'])}")
print(f"{'─'*42}")

### 7b. Augmented training

Enables all four improvements at once: SpecAugment, WeightedRandomSampler,
label smoothing, and progressive unfreezing.

In [ ]:
import json, torch
from pathlib import Path
from src.model import BirdTrainer, TrainingConfig

aug_cfg = TrainingConfig.from_yaml('config/default.yaml')

# ── Augmented run settings ────────────────────────────────────────────────
aug_cfg.checkpoint_name      = 'best_model_aug.pt'
aug_cfg.augment_strategy     = 'specaugment'  # FreqMask + TimeMask + Noise + Erase
aug_cfg.label_smoothing      = 0.1
aug_cfg.use_weighted_sampler = True
aug_cfg.progressive_unfreeze = True
aug_cfg.unfreeze_epoch       = 5

print(f"{'─'*52}")
print(f"  Augmentation   : {aug_cfg.augment_strategy}")
print(f"  Label smoothing: {aug_cfg.label_smoothing}")
print(f"  Weighted sampl.: {aug_cfg.use_weighted_sampler}")
print(f"  Prog. unfreeze : {aug_cfg.progressive_unfreeze}  (epoch {aug_cfg.unfreeze_epoch})")
print(f"  Output         : {aug_cfg.output_dir}/{aug_cfg.checkpoint_name}")
print(f"{'─'*52}\n")

# ── Toggle ────────────────────────────────────────────────────────────────
FORCE_RETRAIN_AUG = False   # set True to retrain even if checkpoint exists
# ─────────────────────────────────────────────────────────────────────────

_ckpt = Path(aug_cfg.output_dir) / aug_cfg.checkpoint_name
_hist = _ckpt.with_suffix('.history.json')

if not FORCE_RETRAIN_AUG and _ckpt.exists():
    print(f'Checkpoint found — skipping augmented training.')
    print(f'  {_ckpt}')
    print(f'  Set FORCE_RETRAIN_AUG = True to retrain.\n')
    aug_trainer         = BirdTrainer(aug_cfg)
    _ckpt_data          = torch.load(_ckpt, map_location='cpu', weights_only=False)
    aug_trainer.classes = _ckpt_data['classes']
    aug_history         = json.loads(_hist.read_text()) if _hist.exists() else {
        'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': [],
        'test_loss': 0.0, 'test_acc': 0.0,
    }
    aug_path = _ckpt
    print(f'Best model : {aug_path}')
    print(f'Test acc   : {aug_history.get("test_acc") or "n/a (run once to record)"}')
else:
    aug_trainer            = BirdTrainer(aug_cfg)
    aug_path, aug_history  = aug_trainer.train(species=SPECIES)
    _hist.write_text(json.dumps(aug_history))
    print(f'\nBest model : {aug_path}')
    print(f'Test acc   : {aug_history["test_acc"]:.3f}')

### 7c. Comparison

In [ ]:
# ── Learning curves comparison ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

b_epochs = list(range(1, len(baseline_history['train_loss']) + 1))
a_epochs = list(range(1, len(aug_history['train_loss'])      + 1))

# Loss
axes[0].plot(b_epochs, baseline_history['val_loss'], 'C0--',  label='baseline val')
axes[0].plot(a_epochs, aug_history['val_loss'],      'C1-',   label='augmented val')
axes[0].plot(b_epochs, baseline_history['train_loss'],'C0:', alpha=0.5, label='baseline train')
axes[0].plot(a_epochs, aug_history['train_loss'],     'C1:', alpha=0.5, label='augmented train')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(fontsize=8)

# Accuracy
axes[1].plot(b_epochs, baseline_history['val_acc'], 'C0--',  label='baseline val')
axes[1].plot(a_epochs, aug_history['val_acc'],      'C1-',   label='augmented val')
axes[1].plot(b_epochs, baseline_history['train_acc'],'C0:', alpha=0.5, label='baseline train')
axes[1].plot(a_epochs, aug_history['train_acc'],     'C1:', alpha=0.5, label='augmented train')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1); axes[1].legend(fontsize=8)

# Overfit gap (train - val accuracy)
b_gap = [tr - vl for tr, vl in zip(baseline_history['train_acc'], baseline_history['val_acc'])]
a_gap = [tr - vl for tr, vl in zip(aug_history['train_acc'],      aug_history['val_acc'])]
axes[2].plot(b_epochs, b_gap, 'C0--', label='baseline')
axes[2].plot(a_epochs, a_gap, 'C1-',  label='augmented')
axes[2].axhline(0, color='gray', linewidth=0.8)
axes[2].set_title('Overfit gap (train acc − val acc)')
axes[2].set_xlabel('Epoch'); axes[2].legend(fontsize=8)

fig.suptitle('Baseline vs Augmented — training comparison', fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# ── Summary metrics table ──────────────────────────────────────────────────
import pandas as pd

rows = [
    {
        'Run':             'Baseline',
        'Augmentation':    'basic',
        'Label smoothing': 0.0,
        'Weighted sampl.': False,
        'Prog. unfreeze':  False,
        'Best val acc':    round(max(baseline_history['val_acc']), 3),
        'Best val loss':   round(min(baseline_history['val_loss']), 4),
        'Test acc':        round(baseline_history['test_acc'],     3),
        'Epochs':          len(baseline_history['train_loss']),
    },
    {
        'Run':             'Augmented',
        'Augmentation':    'specaugment',
        'Label smoothing': 0.1,
        'Weighted sampl.': True,
        'Prog. unfreeze':  True,
        'Best val acc':    round(max(aug_history['val_acc']), 3),
        'Best val loss':   round(min(aug_history['val_loss']), 4),
        'Test acc':        round(aug_history['test_acc'],     3),
        'Epochs':          len(aug_history['train_loss']),
    },
]

df_cmp = pd.DataFrame(rows).set_index('Run')
display(df_cmp)

delta_acc  = aug_history['test_acc'] - baseline_history['test_acc']
delta_sign = '+' if delta_acc >= 0 else ''
print(f"\nTest accuracy delta: {delta_sign}{delta_acc:.3f}"
      f"  ({delta_sign}{delta_acc*100:.1f} pp)")

In [ ]:
# ── Per-species F1 comparison ──────────────────────────────────────────────
from sklearn.metrics import classification_report

b_true, b_pred = baseline_trainer.evaluate(baseline_path, species=SPECIES)
a_true, a_pred = aug_trainer.evaluate(aug_path,           species=SPECIES)

b_report = classification_report(b_true, b_pred, target_names=baseline_trainer.classes,
                                  output_dict=True, zero_division=0)
a_report = classification_report(a_true, a_pred, target_names=aug_trainer.classes,
                                  output_dict=True, zero_division=0)

# Build per-class F1 dataframe
classes = baseline_trainer.classes
df_f1 = pd.DataFrame({
    'Baseline F1':  [round(b_report[c]['f1-score'], 3) for c in classes],
    'Augmented F1': [round(a_report[c]['f1-score'], 3) for c in classes],
}, index=classes)
df_f1['Δ F1'] = (df_f1['Augmented F1'] - df_f1['Baseline F1']).round(3)
df_f1 = df_f1.sort_values('Δ F1')

display(df_f1)

# Bar chart — delta F1 per class
fig, ax = plt.subplots(figsize=(max(10, len(classes) * 0.7), 4))
colors = ['C1' if v >= 0 else 'C3' for v in df_f1['Δ F1']]
ax.bar(range(len(df_f1)), df_f1['Δ F1'], color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(range(len(df_f1)))
ax.set_xticklabels([c.replace('_', '\n') for c in df_f1.index], fontsize=7, rotation=0)
ax.set_ylabel('Δ F1  (augmented − baseline)')
ax.set_title('Per-species F1 improvement  (orange = better, red = worse)')
fig.tight_layout()
plt.show()

---
## 8. Interactive prediction — classify your own recording

Upload any field recording (`.mp3`, `.wav`, `.ogg`, `.flac`) and the model
will identify the bird species.

Pipeline:
1. The audio is resampled and split into `clip_duration`-second windows
2. Each clip is converted to a mel spectrogram using the **exact same pipeline** as training
3. The best available checkpoint (augmented ▶ baseline ▶ saved) runs inference on every clip
4. Predictions are aggregated by **weighted majority vote** across all clips

> **Colab**: the upload widget appears automatically.  
> **Local Jupyter**: set `AUDIO_FILE_PATH = 'your_file.mp3'` in the cell below.

In [ ]:
from pathlib import Path
from IPython.display import display

# ── Option A: provide a local path directly ───────────────────────────────
AUDIO_FILE_PATH = None   # e.g. '/content/my_recording.mp3'

# ── Option B: interactive upload (auto-detected) ──────────────────────────
if AUDIO_FILE_PATH is None:
    try:
        from google.colab import files as _colab_files
        print("Upload your audio file (.mp3 / .wav / .ogg / .flac):")
        _uploaded = _colab_files.upload()
        if _uploaded:
            AUDIO_FILE_PATH = list(_uploaded.keys())[0]
        else:
            print("No file uploaded.")
    except ImportError:
        # Standard Jupyter — try ipywidgets FileUpload
        try:
            import ipywidgets as _widgets
            _upw = _widgets.FileUpload(
                accept='.mp3,.wav,.ogg,.flac', multiple=False,
                description='Upload audio',
            )
            display(_upw)
            print("↑ Select a file above, wait for the upload bar to disappear,")
            print("  then set AUDIO_FILE_PATH = '<filename>' and re-run this cell.")
        except ImportError:
            print("Set  AUDIO_FILE_PATH = 'path/to/your_recording.mp3'  and re-run.")

if AUDIO_FILE_PATH:
    audio_path = Path(AUDIO_FILE_PATH)
    print(f"\nReady : {audio_path.name}  ({audio_path.stat().st_size / 1024:.1f} KB)")
else:
    audio_path = None

In [ ]:
import tempfile, shutil as _shutil_
from collections import Counter
from IPython.display import Audio, display

from src.preprocessing import SpectrogramConverter
from src.model import predict as _predict, load_model as _load_model

if audio_path is None or not audio_path.exists():
    print("No audio file — run the upload cell above first.")
else:
    # ── Select best available checkpoint ──────────────────────────────────
    try:
        _ckpt = aug_path;      _run = 'augmented'
    except NameError:
        try:
            _ckpt = baseline_path; _run = 'baseline'
        except NameError:
            _ckpt = 'models/best_model.pt'; _run = 'saved'

    _, _classes_list, _ = _load_model(str(_ckpt))
    print(f"Model : {_run}  ({_ckpt})")
    print(f"Classes: {len(_classes_list)}\n")

    # ── Play the uploaded audio (first clip_duration seconds) ─────────────
    import librosa as _librosa
    _y, _sr = _librosa.load(str(audio_path), sr=AUDIO_CONFIG.sample_rate, mono=True)
    _clip_s = int(AUDIO_CONFIG.clip_duration * _sr)
    _n_clips = max(1, len(_y) // _clip_s)
    print(f"Duration : {len(_y) / _sr:.1f}s  →  {_n_clips} clip(s) of {AUDIO_CONFIG.clip_duration}s")
    print("Audio preview (first clip):")
    display(Audio(_y[:_clip_s], rate=_sr))

    # ── Generate PNGs via the training pipeline (temp directory) ──────────
    _tmproot = tempfile.mkdtemp()
    try:
        _sp_dir  = _tmproot + '/user_upload'
        _png_dir = _tmproot + '/pngs'
        import os; os.makedirs(_sp_dir); os.makedirs(_png_dir)
        _shutil_.copy(audio_path, _sp_dir + '/' + audio_path.name)

        _conv = SpectrogramConverter(output_dir=_png_dir, config=AUDIO_CONFIG)
        _pngs = _conv.convert_file(audio_path, _png_dir)

        # ── Show mel spectrogram of first clip ────────────────────────────
        fig, ax = plt.subplots(figsize=(8, 3))
        try:
            _conv.plot_spectrogram(audio_path, clip_index=0, ax=ax,
                                   title=audio_path.stem)
        except Exception as _e:
            ax.text(0.5, 0.5, f'Preview unavailable\n{_e}',
                    ha='center', va='center', transform=ax.transAxes)
        plt.tight_layout(); plt.show()

        # ── Inference — one forward pass per clip ─────────────────────────
        if not _pngs:
            print("No spectrograms generated — check the audio file format.")
        else:
            print(f"Clips generated: {len(_pngs)}  — running inference...")
            _votes = Counter()
            for _png in _pngs:
                for _cls, _p in _predict(str(_png), str(_ckpt),
                                          device='cpu',
                                          top_k=len(_classes_list)):
                    _votes[_cls] += _p

            _total  = sum(_votes.values())
            _ranked = sorted(_votes.items(), key=lambda x: x[1], reverse=True)

            print(f"\nTop predictions  ({len(_pngs)} clip(s) analysed):")
            print(f"{'─'*54}")
            for _rank, (_cls, _w) in enumerate(_ranked[:5], 1):
                _pct = _w / _total * 100
                _bar = '█' * int(_pct / 2)
                print(f"  {_rank}. {_cls.replace('_', ' '):<33} {_pct:5.1f}%  {_bar}")
            print(f"{'─'*54}")
            print(f"\n  Classified as: {_ranked[0][0].replace('_', ' ')}")
    finally:
        _shutil_.rmtree(_tmproot, ignore_errors=True)